Electrode map generator
this interface automatically reads electrode layouts from dxf files,
and maps channels to PCB channels.

return: 3 x n_channels matrix (x-location, y-loction, PCB channel)

In [1]:
import numpy as np, matplotlib.pyplot as plt, ezdxf, csv
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import utils

In [7]:
doc = ezdxf.readfile("../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf")
doc = ezdxf.readfile("../electrode_map_example/example2/Xiaomin_electrode_v2.dxf")
layers = sorted({e.dxf.layer for e in doc.modelspace()})
print(layers)

['electrodes', 'etching', 'metalization']


In [3]:
# give the layer name of the electrode contours in the DXF file
# if you don't know, give None and instead the number of contours to extract (the largest ones will be used)
layer_name = "electrode"
# if you don't know the layer name, give None and instead the number of contours to extract (the one closest to the origin will be used)
# ATTENtiON: we assume that the last contour is a hook loop, which we will remove from the electrode layout
num_contours = 64

In [5]:
dxf_chooser = FileChooser(filter_pattern="*.dxf")
csv_chooser = FileChooser(filter_pattern="*.csv")
display(dxf_chooser, csv_chooser)

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

In [6]:
run_button = widgets.Button(description="Generate Channel Map", button_style="success")
output = widgets.Output()
# Store channel map and filenames for export
state = {"channel_map": None, "dxf_file": None, "csv_file": None}

# Modify on_run_clicked to save state
def on_run_clicked(b):
    with output:
        output.clear_output(wait=True)
        
        if not dxf_chooser.value:
            print("❌ Please select a DXF file")
            return
        if not csv_chooser.value:
            print("❌ Please select a CSV file")
            return
        
        try:
            print("Loading electrode layout...")
            xy = utils.load_and_plot_electrodes(dxf_chooser.value, layer_name=layer_name, num_contours=num_contours)
            
            print("\nCreating channel map...")
            channel_map = utils.create_channel_map(xy, csv_chooser.value)
            
            # Store for export
            state["channel_map"] = channel_map
            state["dxf_file"] = dxf_chooser.value
            state["csv_file"] = csv_chooser.value
            
            print(f"\n✓ Success!")
            print(f"  Shape: {channel_map.shape}")
            print(f"  Sample column: {channel_map[:, 0]}")
        except Exception as e:
            print(f"❌ Error: {e}")

run_button.on_click(on_run_clicked)
display(run_button, output)

Button(button_style='success', description='Generate Channel Map', style=ButtonStyle())

Output()

In [8]:
run_button.close()
output.clear_output()

In [9]:
# Export button
export_button = widgets.Button(description="Export to CSV", button_style="info")
export_output = widgets.Output()

def on_export_clicked(b):
    with export_output:
        export_output.clear_output(wait=True)
        
        if state["channel_map"] is None:
            print("❌ Generate a channel map first")
            return
        
        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"], state["dxf_file"])
            print(f"✓ Exported to: {export_file}")
        except Exception as e:
            print(f"❌ Export failed: {e}")

export_button.on_click(on_export_clicked)
display(export_button, export_output)

Button(button_style='info', description='Export to CSV', style=ButtonStyle())

Output()